# MIMIC CSV Walkthrough

This notebook shows the CSV files created for the MIMIC cohorts, how to inspect them, and how they are expected to work with the `tabular_gia` pipeline.

It covers:
- ICU-stay cohort (`mimic_iv_icu_mortality.csv`)
- Admission cohort (`mimic_iv_admission_mortality.csv`)
- Strict admission cohort (no post-admission ICU features)
- Correlation CSV outputs in `tabular_gia/results/examples`


In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import yaml
import matplotlib.pyplot as plt


def find_repo_root(start: Path) -> Path:
    cur = start.resolve()
    for p in [cur] + list(cur.parents):
        if (p / "tabular_gia").exists():
            return p
    raise RuntimeError("Could not find repo root containing 'tabular_gia'")


ROOT = find_repo_root(Path.cwd())
TG = ROOT / "tabular_gia"
print("Repo root:", ROOT)
print("tabular_gia:", TG)


Repo root: C:\Users\maxim\Documents\GitHub\LeakPro
tabular_gia: C:\Users\maxim\Documents\GitHub\LeakPro\tabular_gia


## 1) Key CSV files

These are the main cohort CSVs and their YAML metadata files.


In [2]:
paths = {
    "icu_csv": TG / "data/binary/mimic_iv_icu_mortality/mimic_iv_icu_mortality.csv",
    "icu_yaml": TG / "data/binary/mimic_iv_icu_mortality/mimic_iv_icu_mortality.yaml",
    "adm_csv": TG / "data/binary/mimic_iv_admission_mortality/mimic_iv_admission_mortality.csv",
    "adm_yaml": TG / "data/binary/mimic_iv_admission_mortality/mimic_iv_admission_mortality.yaml",
    "adm_strict_csv": TG / "data/binary/mimic_iv_admission_mortality_strict/mimic_iv_admission_mortality.csv",
    "adm_strict_yaml": TG / "data/binary/mimic_iv_admission_mortality_strict/mimic_iv_admission_mortality.yaml",
}

for k, v in paths.items():
    print(f"{k:15s} exists={v.exists()}  -> {v}")


icu_csv         exists=True  -> C:\Users\maxim\Documents\GitHub\LeakPro\tabular_gia\data\binary\mimic_iv_icu_mortality\mimic_iv_icu_mortality.csv
icu_yaml        exists=True  -> C:\Users\maxim\Documents\GitHub\LeakPro\tabular_gia\data\binary\mimic_iv_icu_mortality\mimic_iv_icu_mortality.yaml
adm_csv         exists=True  -> C:\Users\maxim\Documents\GitHub\LeakPro\tabular_gia\data\binary\mimic_iv_admission_mortality\mimic_iv_admission_mortality.csv
adm_yaml        exists=True  -> C:\Users\maxim\Documents\GitHub\LeakPro\tabular_gia\data\binary\mimic_iv_admission_mortality\mimic_iv_admission_mortality.yaml
adm_strict_csv  exists=True  -> C:\Users\maxim\Documents\GitHub\LeakPro\tabular_gia\data\binary\mimic_iv_admission_mortality_strict\mimic_iv_admission_mortality.csv
adm_strict_yaml exists=True  -> C:\Users\maxim\Documents\GitHub\LeakPro\tabular_gia\data\binary\mimic_iv_admission_mortality_strict\mimic_iv_admission_mortality.yaml


## 2) Quick dataset summary

For each cohort, show rows, columns, target rate, and first rows.


In [3]:
def summarize_dataset(csv_path: Path, target: str = "hospital_expire_flag"):
    df = pd.read_csv(csv_path)
    summary = {
        "file": csv_path.name,
        "rows": len(df),
        "cols": len(df.columns),
        "target_exists": target in df.columns,
        "target_rate": float(df[target].mean()) if target in df.columns else None,
        "missing_total": int(df.isna().sum().sum()),
    }
    return df, summary


cohorts = {
    "ICU": paths["icu_csv"],
    "Admission": paths["adm_csv"],
    "AdmissionStrict": paths["adm_strict_csv"],
}

frames = {}
for name, p in cohorts.items():
    df, s = summarize_dataset(p)
    frames[name] = df
    print("\n", name, "summary")
    for k, v in s.items():
        print(f"  {k}: {v}")
    display(df.head(3))



 ICU summary
  file: mimic_iv_icu_mortality.csv
  rows: 94458
  cols: 17
  target_exists: True
  target_rate: 0.12015922420546699
  missing_total: 64662


,hospital_expire_flag,gender,age_at_admit_est,admit_to_icu_hours,admit_hour,admit_weekday,had_ed,ed_los_hours,ed_to_admit_hours,admission_type,admission_location,insurance,language,marital_status,race,first_careunit,anchor_year_group
0,0,F,52,1.416667,12,6,1,8.100000,6.683333,EW EMER.,EMERGENCY ROOM,Medicaid,English,WIDOWED,WHITE,Medical Intensive Care Unit (MICU),2014 - 2016
1,0,F,86,1.583333,18,0,1,7.933333,6.350000,EW EMER.,EMERGENCY ROOM,Medicare,English,WIDOWED,WHITE,Medical Intensive Care Unit (MICU),2008 - 2010
2,0,F,76,1.066667,7,5,1,2.283333,1.216667,EW EMER.,EMERGENCY ROOM,Medicare,English,MARRIED,BLACK/AFRICAN AMERICAN,Medical Intensive Care Unit (MICU),2008 - 2010



 Admission summary
  file: mimic_iv_admission_mortality.csv
  rows: 546028
  cols: 18
  target_exists: True
  target_rate: 0.02161244478305142
  missing_total: 794362


,hospital_expire_flag,gender,age_at_admit_est,admit_hour,admit_weekday,had_ed,ed_los_hours,ed_to_admit_hours,admission_type,admission_location,insurance,language,marital_status,race,anchor_year_group,has_icu,admit_to_first_icu_hours,first_icu_careunit
0,0,F,52,22,5,1,4.216667,3.100000,URGENT,TRANSFER FROM HOSPITAL,Medicaid,English,WIDOWED,WHITE,2014 - 2016,0,NaN,UNKNOWN
1,0,F,52,18,0,1,5.616667,2.550000,EW EMER.,EMERGENCY ROOM,Medicaid,English,WIDOWED,WHITE,2014 - 2016,0,NaN,UNKNOWN
2,0,F,52,23,5,1,4.766667,2.766667,EW EMER.,EMERGENCY ROOM,Medicaid,English,WIDOWED,WHITE,2014 - 2016,0,NaN,UNKNOWN



 AdmissionStrict summary
  file: mimic_iv_admission_mortality.csv
  rows: 546028
  cols: 15
  target_exists: True
  target_rate: 0.02161244478305142
  missing_total: 333576


,hospital_expire_flag,gender,age_at_admit_est,admit_hour,admit_weekday,had_ed,ed_los_hours,ed_to_admit_hours,admission_type,admission_location,insurance,language,marital_status,race,anchor_year_group
0,0,F,52,22,5,1,4.216667,3.100000,URGENT,TRANSFER FROM HOSPITAL,Medicaid,English,WIDOWED,WHITE,2014 - 2016
1,0,F,52,18,0,1,5.616667,2.550000,EW EMER.,EMERGENCY ROOM,Medicaid,English,WIDOWED,WHITE,2014 - 2016
2,0,F,52,23,5,1,4.766667,2.766667,EW EMER.,EMERGENCY ROOM,Medicaid,English,WIDOWED,WHITE,2014 - 2016


## 3) Compare feature sets across cohorts

This highlights what was removed/added between the cohorts.


In [4]:
icu_cols = set(frames["ICU"].columns)
adm_cols = set(frames["Admission"].columns)
strict_cols = set(frames["AdmissionStrict"].columns)

print("Columns only in Admission (not strict):")
print(sorted(adm_cols - strict_cols))

print("\nColumns only in Strict (not admission):")
print(sorted(strict_cols - adm_cols))

print("\nColumns in ICU not in Admission:")
print(sorted(icu_cols - adm_cols)[:30], "..." if len(icu_cols - adm_cols) > 30 else "")


Columns only in Admission (not strict):
['admit_to_first_icu_hours', 'first_icu_careunit', 'has_icu']

Columns only in Strict (not admission):
[]

Columns in ICU not in Admission:
['admit_to_icu_hours', 'first_careunit'] 


## 4) Validate YAML metadata against CSV

The dataloader expects:
- `target`
- `missing_values`
- `numerical_columns`
- `categorical_columns`

This cell checks that metadata columns exist in each CSV.


In [5]:
def load_yaml(path: Path):
    with open(path, "r", encoding="utf-8") as f:
        return yaml.safe_load(f)


def validate_meta(csv_path: Path, yaml_path: Path):
    df = pd.read_csv(csv_path)
    meta = load_yaml(yaml_path)
    cols = set(df.columns)

    target = meta.get("target")
    num_cols = meta.get("numerical_columns", [])
    cat_cols = meta.get("categorical_columns", [])

    missing_in_csv = [c for c in ([target] + num_cols + cat_cols) if c not in cols]
    overlap_num_cat = sorted(set(num_cols).intersection(set(cat_cols)))

    report = {
        "target": target,
        "n_num_cols": len(num_cols),
        "n_cat_cols": len(cat_cols),
        "missing_in_csv": missing_in_csv,
        "num_cat_overlap": overlap_num_cat,
    }
    return report, meta


pairs = [
    ("ICU", paths["icu_csv"], paths["icu_yaml"]),
    ("Admission", paths["adm_csv"], paths["adm_yaml"]),
    ("AdmissionStrict", paths["adm_strict_csv"], paths["adm_strict_yaml"]),
]

for name, csv_p, yaml_p in pairs:
    report, meta = validate_meta(csv_p, yaml_p)
    print(f"\n{name}")
    for k, v in report.items():
        print(f"  {k}: {v}")



ICU
  target: hospital_expire_flag
  n_num_cols: 6
  n_cat_cols: 10
  missing_in_csv: []
  num_cat_overlap: []

Admission
  target: hospital_expire_flag
  n_num_cols: 6
  n_cat_cols: 11
  missing_in_csv: []
  num_cat_overlap: []

AdmissionStrict
  target: hospital_expire_flag
  n_num_cols: 5
  n_cat_cols: 9
  missing_in_csv: []
  num_cat_overlap: []


## 5) Correlation CSV files

Load the generated correlation outputs and preview their content.


In [6]:
corr_files = [
    TG / "results/examples/mimic_icu_numeric_corr_full.csv",
    TG / "results/examples/mimic_icu_top_target_corr_top100.csv",
    TG / "results/examples/mimic_admission_numeric_corr_full.csv",
    TG / "results/examples/mimic_admission_top_target_corr_top100.csv",
    TG / "results/examples/mimic_admission_strict_numeric_corr_full.csv",
    TG / "results/examples/mimic_admission_strict_top_target_corr_top100.csv",
]

for p in corr_files:
    print(f"{p.name:50s} exists={p.exists()}")

print("\nTop target correlations (strict admission):")
strict_top = pd.read_csv(TG / "results/examples/mimic_admission_strict_top_target_corr_top100.csv")
display(strict_top.head(20))


mimic_icu_numeric_corr_full.csv                    exists=True
mimic_icu_top_target_corr_top100.csv               exists=True
mimic_admission_numeric_corr_full.csv              exists=True
mimic_admission_top_target_corr_top100.csv         exists=True
mimic_admission_strict_numeric_corr_full.csv       exists=True
mimic_admission_strict_top_target_corr_top100.csv  exists=True

Top target correlations (strict admission):


,Unnamed: 0,corr
0,marital_status_UNKNOWN,0.108030
1,race_UNKNOWN,0.104718
2,age_at_admit_est,0.094979
3,admission_location_TRANSFER FROM HOSPITAL,0.088920
4,admission_type_EW EMER.,0.075686
5,admission_type_EU OBSERVATION,-0.075300
6,admission_location_TRANSFER FROM SKILLED NURSI...,0.073430
7,ed_los_hours,-0.068469
8,insurance_Medicare,0.057697
9,admission_location_PHYSICIAN REFERRAL,-0.053640


## 6) Visualize one correlation matrix from CSV

This reads the saved numeric correlation CSV and plots it.


In [ ]:
corr_df = pd.read_csv(TG / "results/examples/mimic_admission_strict_numeric_corr_full.csv", index_col=0)

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(corr_df.values, vmin=-1, vmax=1, cmap="coolwarm")
ax.set_xticks(range(len(corr_df.columns)))
ax.set_xticklabels(corr_df.columns, rotation=60, ha="right", fontsize=8)
ax.set_yticks(range(len(corr_df.index)))
ax.set_yticklabels(corr_df.index, fontsize=8)
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
ax.set_title("Admission Strict: numeric correlation matrix")
fig.tight_layout()
plt.show()


## 7) How these CSVs are supposed to work in `tabular_gia`

Expected pattern:
1. One row per sample (ICU stay or admission, depending on cohort)
2. Binary target column: `hospital_expire_flag`
3. Metadata YAML lists numeric and categorical feature columns
4. `tabular_gia/fl/dataloader/tabular_dataloader.py` reads CSV + YAML, preprocesses, one-hot encodes categoricals, and builds client dataloaders

To switch active dataset for training/sweeps, update:
- `tabular_gia/configs/dataset/dataset.yaml`
